In [2]:
import sqlite3
import pandas as pd

In [40]:
query = '''
SELECT *
FROM employees
WHERE vacation_hours > 40;
'''
with sqlite3.connect('employees.db') as conn:
    df = pd.read_sql_query(query, conn)

df['customer_id'] = pd.to_numeric(df['customer_id'])
df.head()


,customer_id,email_address,vacation_hours,sallaried,title
0,2,keith0@adventure-works.com,80,1,manager
1,3,donna0@adventure-works.com,69,1,sales rep
2,4,janet1@adventure-works.com,49,1,sales rep
3,5,lucy0@adventure-works.com,49,1,sales rep
4,6,rosmarie0@adventure-works.com,88,1,manager


In [41]:
df['employee_name'] = df['email_address'].str.extract(r'([a-z]+)(?=\d)')
df.to_excel('out/employees_with_high_vaca_hours.xlsx', index=False)
df.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 554 entries, 0 to 553
Data columns (total 6 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   customer_id     554 non-null    int64 
 1   email_address   554 non-null    object
 2   vacation_hours  554 non-null    int64 
 3   sallaried       554 non-null    int64 
 4   title           554 non-null    object
 5   employee_name   554 non-null    object
dtypes: int64(3), object(3)
memory usage: 26.1+ KB


HR Question 2

In [ ]:
query = '''
SELECT *
FROM employees
'''
with sqlite3.connect('employees.db') as conn:
    employee_df = pd.read_sql_query(query, conn)

employee_df.head()

,customer_id,email_address,vacation_hours,sallaried,title
0,1,orlando0@adventure-works.com,18,0,sales rep
1,2,keith0@adventure-works.com,80,1,manager
2,3,donna0@adventure-works.com,69,1,sales rep
3,4,janet1@adventure-works.com,49,1,sales rep
4,5,lucy0@adventure-works.com,49,1,sales rep


In [ ]:
employee_df_grouped = employee_df.groupby(['sallaried', 'title']).mean('vacation_hours')
employee_df_grouped.head()
employee_df.to_excel('out/avg_PTO.xlsx', index=False)

HR Question 3

In [36]:
customer_sales_orders_table = pd.read_csv('customer_sales_orders_tables.csv')
query = '''
SELECT SalesOrderID, OrderQty
FROM SalesOrderDetail
'''
with sqlite3.connect('AdventureWorks.db') as conn:
    sales_order_qty = pd.read_sql_query(query, conn)

sales_order_qty_total = sales_order_qty.groupby('SalesOrderID')['OrderQty'].sum()

employee_id_separated_qty = pd.merge(customer_sales_orders_table, sales_order_qty_total, right_on='SalesOrderID', left_on='sales_order_ids')

employee_id_total_qty = employee_id_separated_qty.groupby('customer_rep_ids')['OrderQty'].sum()
employee_id_total_qty.head()

customer_rep_ids
29486    173
29492    582
29494    173
29496    217
29499    173
Name: OrderQty, dtype: int64

In [45]:
employee_name_qty = pd.merge(df, employee_id_total_qty, left_on='customer_id', right_on='customer_rep_ids')
print(employee_name_qty.sample(10))
employee_name_qty.to_excel('out/total_sales_by_rep.xlsx', index=False)

     customer_id                  email_address  vacation_hours  sallaried  \
135        29911       tim0@adventure-works.com              71          1   
21         29568    donald0@adventure-works.com              79          1   
90         29789     brian4@adventure-works.com              43          0   
175        30018  virginia0@adventure-works.com              87          1   
0          29492       jay1@adventure-works.com              56          1   
68         29727      bart0@adventure-works.com              46          0   
12         29530    daniel0@adventure-works.com              97          1   
176        30021       ben1@adventure-works.com              67          0   
146        29932   rebecca2@adventure-works.com              75          1   
169        29998      jane3@adventure-works.com              44          1   

         title employee_name  OrderQty  
135  sales rep           tim       167  
21     manager        donald       428  
90     manager    